In [ ]:
import awswrangler as wr
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. CARGA OPTIMIZADA (Usamos un límite para no colapsar la memoria)
# Si tu tabla Gold es gigante, el 'LIMIT' es obligatorio para que el EDA no se cuelgue.
print("Cargando datos desde Gold...")
query = 'SELECT * FROM "data_lake_academico_gold"."fact_estudiante_periodo" LIMIT 10000'
df_gold = wr.athena.read_sql_query(query, database="data_lake_academico_gold")

# --- ANÁLISIS 1: CALIDAD DE DATOS ---
# Auditamos nulos inmediatamente
print("--- Auditoría de Nulos ---")
print(df_gold.isnull().sum() / len(df_gold) * 100)

# --- ANÁLISIS 2: TASA DE DESERCIÓN POR ESCUELA ---
plt.figure(figsize=(12, 6))
sns.barplot(data=df_gold, x='escuela', y='desertion_t1', 
            estimator=lambda x: sum(x)/len(x), hue='escuela', palette='viridis', legend=False)
plt.title('Tasa de Deserción por Escuela (Promedio)')
plt.ylabel('Tasa de Deserción')
plt.xticks(rotation=45)
plt.show()

# --- ANÁLISIS 3: IMPACTO DE LA FIDELIDAD ---
plt.figure(figsize=(10, 6))
# Fíjate que añadí hue='desertion_t1' dentro de los argumentos
sns.boxplot(data=df_gold, x='desertion_t1', y='tasa_permanencia', 
            palette='Set2', hue='desertion_t1', legend=False)
plt.title('Distribución de Tasa de Permanencia según Deserción')
plt.show()

# --- ANÁLISIS 4: CORRELACIÓN ---
# Filtramos solo columnas numéricas para el mapa de calor
num_cols = df_gold.select_dtypes(include=['float64', 'int64']).columns
plt.figure(figsize=(12, 10))
sns.heatmap(df_gold[num_cols].corr(), annot=True, cmap='coolwarm', fmt=".1f")
plt.title('Mapa de Calor: Correlaciones en Capa Gold')
plt.show()

# --- ANÁLISIS 5: BALANCE DE CLASES ---
# Esto es vital para saber si tu modelo sufrirá por desbalance
plt.figure(figsize=(6, 4))
sns.countplot(data=df_gold, x='desertion_t1', hue='desertion_t1', palette='pastel', legend=False)
plt.title('Distribución de Estudiantes (0=Continúa, 1=Deserta)')
plt.show()